# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library, referencing dataset entities by their `@id`.

### Dataset Source
The dataset source is described with a Croissant schema and can be accessed via the URL below.

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. The actual dataset will be loaded based on its Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# URL of the Croissant schema JSON-LD
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access and display main metadata fields
meta = dataset.metadata
print(f"{meta.name}: {meta.description}\n")
print(f"Version: {getattr(meta, 'version', 'N/A')}")
print(f"Published: {getattr(meta, 'datePublished', 'N/A')}")
print(f"License: {getattr(meta, 'license', 'N/A')}")
print(f"Cite as: {getattr(meta, 'citeAs', 'N/A')}")

## 2. Data Overview
Let's review available record sets, fields (columns), and their `@id` values. This prepares us to reference data precisely in subsequent steps.

In [ ]:
# List all record sets and their @id, along with fields and columns' @id
def print_record_sets_overview(dataset):
    if not hasattr(dataset.metadata, 'recordSet') or not dataset.metadata.recordSet:
        print("No record sets found in the metadata.")
        return []
    record_sets = dataset.metadata.recordSet
    # record_sets could be a list or a dict (single), normalize to list
    if isinstance(record_sets, dict):
        record_sets = [record_sets]
    record_set_ids = []
    for recset in record_sets:
        recset_id = recset.get('@id', None)
        recset_name = recset.get('name', '(no name)')
        print(f"Record set name: {recset_name}  |  @id: {recset_id}")
        record_set_ids.append(recset_id)
        if 'field' in recset:
            fields = recset['field']
            if isinstance(fields, dict):
                fields = [fields]
            for field in fields:
                field_id = field.get('@id', None)
                field_name = field.get('name', '(no name)')
                print(f"  Field: {field_name}  |  @id: {field_id}")
                if 'column' in field:
                    cols = field['column']
                    if isinstance(cols, dict):
                        cols = [cols]
                    for col in cols:
                        col_id = col.get('@id', None)
                        col_name = col.get('name', '(no name)')
                        print(f"    Column: {col_name}  |  @id: {col_id}")
    return record_set_ids

record_set_ids = print_record_sets_overview(dataset)
if not record_set_ids:
    print("Trying to infer record sets dynamically from schema...")
    # alternatively, try to list available record_set IDs via dataset.records
    hint = getattr(dataset, 'records', None)
    if hint:
        try:
            # Try to enumerate possible record sets (uses internal mechanism)
            # If not available, we will inform the user
            print("Available record sets may be enumerated directly using the Croissant metadata JSON if needed.")
        except Exception as e:
            print(f"Could not enumerate record sets: {e}")

## 3. Data Extraction
Let's extract data from a specific record set. In this dataset, the main record set typically includes protein quantification/identification table data. We will use the record set and field `@id` values from the overview above.

**Note:** If no record sets are listed, use documentation or examine dataset content to identify the main `@id`. Common defaults include `'cr:RecordSet_0'` or similar canonical IDs. (See documentation or inspect dataset.metadata.recordSet as shown above.)

In [ ]:
# Here we attempt to load all record sets. Replace these IDs in practice with those printed in the overview above.

if not record_set_ids:
    # If no record sets found in the metadata, try canonical defaults
    record_set_ids = ['cr:RecordSet_0']  # Replace with actual record set @id if known

dataframes = {}
for record_set_id in record_set_ids:
    try:
        print(f"Loading records for record set @id='{record_set_id}' ...")
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns in '{record_set_id}': {df.columns.tolist()}")
        display(df.head())
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
We can now apply common processing and exploration steps: filtering, normalization, and grouping data based on field `@id`. For demonstration, we'll pick likely-numeric fields (such as an "abundance" or "peptide count" column) as seen in the extracted DataFrame's columns list above.

In [ ]:
# Choose record set for EDA
record_set_id = record_set_ids[0]  # Replace with actual @id if you have multiple

# List all column IDs
print(f"Columns available in record set '{record_set_id}':\n", dataframes[record_set_id].columns.tolist())

# Select a numeric field by its column name (example: 'PeptideCount' or similar; update as needed)
# Use the actual name from your columns list
numeric_field = None
for col in dataframes[record_set_id].columns:
    # Heuristically select likely numeric fields
    if any(word in col.lower() for word in ['count', 'number', 'abundance', 'mw', 'weight', 'coverage', 'pi']):
        numeric_field = col
        break
if numeric_field is None:
    numeric_field = dataframes[record_set_id].columns[0]  # fallback, pick the first column

print(f"Chosen numeric field for EDA: {numeric_field}")

# Filter for records where numeric_field > threshold
threshold = 10
filtered_df = dataframes[record_set_id]
try:
    filtered_df = filtered_df[pd.to_numeric(filtered_df[numeric_field], errors='coerce') > threshold]
except Exception as e:
    print(f"Warning: Could not apply numeric conversion/filter. Exception: {e}")

print(f"Filtered records with {numeric_field} > {threshold}:")
display(filtered_df.head())

# Normalize the numeric field
try:
    filtered_df[f"{numeric_field}_normalized"] = (
        pd.to_numeric(filtered_df[numeric_field], errors='coerce') - pd.to_numeric(filtered_df[numeric_field], errors='coerce').mean()
    ) / pd.to_numeric(filtered_df[numeric_field], errors='coerce').std()
    print(f"Normalized '{numeric_field}':")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
except Exception as e:
    print(f"Could not normalize field: {e}")

# Try grouping by another likely categorical field
group_field = None
for col in dataframes[record_set_id].columns:
    if any(word in col.lower() for word in ['group', 'sample', 'type', 'class', 'condition']):
        group_field = col
        break
if group_field:
    print(f"Grouping by field: {group_field}")
    grouped_df = filtered_df.groupby(group_field).mean() if group_field in filtered_df else None
    if grouped_df is not None:
        display(grouped_df.head())

## 5. Visualization
Let's plot the distribution of the chosen numeric field and display a boxplot for group-wise means if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(8, 4))
sns.histplot(pd.to_numeric(filtered_df[numeric_field], errors='coerce').dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

# Boxplot grouped by group_field if available
if group_field and group_field in filtered_df:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=filtered_df[group_field], y=pd.to_numeric(filtered_df[numeric_field], errors='coerce'))
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
We have successfully loaded the FAIR² dataset, explored its record set(s), and performed basic exploratory and visualization tasks using referenced `@id` values. For deeper analysis, further domain knowledge about the field semantics and Croissant schema structure is recommended.